<a href="https://colab.research.google.com/github/vladmsnk/dl_aith/blob/contest1/contest2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import zipfile
import os

zip_file_path = '/content/aith-dl-competition-tabular-data-2025-2.zip'
output_dir = './'

os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

print(f"'{zip_file_path}' unzipped to '{output_dir}' successfully.")

'/content/aith-dl-competition-tabular-data-2025-2.zip' unzipped to './' successfully.


In [4]:
import pandas as pd

df = pd.read_csv('/content/train.csv')

In [5]:
df.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,smoking
0,0,35.0,175.0,75.0,86.5,1.2,1.2,1.0,1.0,127.0,...,58.0,108.0,15.6,1.0,0.9,17.0,14.0,21.0,0.0,0.0
1,1,45.0,155.0,60.0,82.0,1.2,1.0,1.0,1.0,129.0,...,50.0,110.0,14.0,1.0,0.7,22.0,18.0,14.0,0.0,0.0
2,2,35.0,175.0,60.0,74.0,1.2,1.2,1.0,1.0,100.0,...,58.0,116.0,14.8,1.0,0.9,20.0,15.0,16.0,0.0,1.0
3,3,60.0,160.0,55.0,74.0,1.2,1.5,1.0,1.0,139.0,...,73.0,95.0,15.1,1.0,0.7,47.0,31.0,15.0,0.0,0.0
4,4,40.0,160.0,55.0,71.0,0.9,1.2,1.0,1.0,100.0,...,66.0,103.0,13.1,1.0,0.6,24.0,21.0,13.0,0.0,0.0


In [6]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


@dataclass
class TabPreprocessor:
    """
    Preprocessor для табличных данных под нейросеть:
    - делит фичи на numeric и categorical
    - для numeric: median-impute + StandardScaler
    - для categorical: value -> index, чтобы создать слой Embedding
    """

    target_col: str = "smoking"
    id_col: str = "id"

    cat_cols: Optional[List[str]] = None
    num_cols: Optional[List[str]] = None

    cat_unique_threshold: int = 20

    medians_: Optional[pd.Series] = None
    scaler_: Optional[StandardScaler] = None
    cat_maps_: Optional[Dict[str, Dict[str, int]]] = None
    cat_cardinalities_: Optional[List[int]] = None

    def fit(self, df: pd.DataFrame) -> "TabPreprocessor":
        df_feat = self._get_features_df(df)

        if self.cat_cols is None or self.num_cols is None:
            inferred_cat = [
                c for c in df_feat.columns
                if df_feat[c].nunique(dropna=True) <= self.cat_unique_threshold
            ]
            inferred_num = [c for c in df_feat.columns if c not in inferred_cat]

            if self.cat_cols is None:
                self.cat_cols = inferred_cat
            if self.num_cols is None:
                self.num_cols = inferred_num

        self.cat_maps_ = {}
        self.cat_cardinalities_ = []
        for c in self.cat_cols:
            vals = df_feat[c].fillna("__NA__").astype(str).unique().tolist()
            mapping = {v: i + 1 for i, v in enumerate(vals)}  # 0 reserved for unknown
            self.cat_maps_[c] = mapping
            self.cat_cardinalities_.append(len(mapping) + 1)

        if self.num_cols:
            num = df_feat[self.num_cols].copy()
            self.medians_ = num.median(numeric_only=True)
            num = num.fillna(self.medians_).values.astype(np.float32)

            self.scaler_ = StandardScaler()
            self.scaler_.fit(num)
        else:
            self.medians_ = None
            self.scaler_ = None

        return self

    def transform(self, df: pd.DataFrame) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        if self.cat_cols is None or self.num_cols is None:
            raise RuntimeError("Preprocessor is not fitted: cat_cols/num_cols are None.")
        if self.cat_maps_ is None or self.cat_cardinalities_ is None:
            raise RuntimeError("Preprocessor is not fitted: cat_maps_/cat_cardinalities_ are None.")
        if self.num_cols and (self.medians_ is None or self.scaler_ is None):
            raise RuntimeError("Preprocessor is not fitted: numeric stats are missing.")

        df_feat = self._get_features_df(df)

        X_cat = None
        if self.cat_cols:
            X_cat = np.zeros((len(df_feat), len(self.cat_cols)), dtype=np.int64)
            for j, c in enumerate(self.cat_cols):
                m = self.cat_maps_[c]
                col = df_feat[c].fillna("__NA__").astype(str).values
                X_cat[:, j] = np.array([m.get(v, 0) for v in col], dtype=np.int64)

        X_num = None
        if self.num_cols:
            num = df_feat[self.num_cols].copy()
            num = num.fillna(self.medians_)
            X_num = self.scaler_.transform(num.values.astype(np.float32)).astype(np.float32)

        return X_num, X_cat

    def fit_transform(self, df: pd.DataFrame) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        return self.fit(df).transform(df)

    def _get_features_df(self, df: pd.DataFrame) -> pd.DataFrame:
        cols_drop = []
        if self.target_col in df.columns:
            cols_drop.append(self.target_col)
        if self.id_col in df.columns:
            cols_drop.append(self.id_col)
        return df.drop(columns=cols_drop, errors="ignore")


In [8]:
%%capture
!pip install pytorch-lightning

In [10]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl


class TabularDataset(Dataset):
    def __init__(self, X_num: np.ndarray | None, X_cat: np.ndarray | None, y: np.ndarray | None = None):
        self.X_num = torch.from_numpy(X_num).float() if X_num is not None else None
        self.X_cat = torch.from_numpy(X_cat).long() if X_cat is not None else None
        self.y = torch.from_numpy(y).float() if y is not None else None

        n = None
        if self.X_num is not None:
            n = self.X_num.shape[0]
        if self.X_cat is not None:
            n = self.X_cat.shape[0] if n is None else n
        if self.y is not None:
            n = self.y.shape[0] if n is None else n

        self.n = n if n is not None else 0

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x_num = self.X_num[idx] if self.X_num is not None else torch.empty(0, dtype=torch.float32)
        x_cat = self.X_cat[idx] if self.X_cat is not None else torch.empty(0, dtype=torch.long)

        if self.y is None:
            return x_num, x_cat

        return x_num, x_cat, self.y[idx]


class SmokingDataModule(pl.LightningDataModule):
    def __init__(
        self,
        df_train: pd.DataFrame,
        df_val: pd.DataFrame,
        df_test: pd.DataFrame | None = None,
        target_col: str = "smoking",
        id_col: str = "id",
        batch_size: int = 512,
        num_workers: int = 2,
        cat_cols: list[str] | None = None,
        num_cols: list[str] | None = None,
        cat_unique_threshold: int = 20,
    ):
        super().__init__()
        self.df_train = df_train
        self.df_val = df_val
        self.df_test = df_test

        self.target_col = target_col
        self.id_col = id_col
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.cat_cols = cat_cols
        self.num_cols = num_cols
        self.cat_unique_threshold = cat_unique_threshold

        self.prep: TabPreprocessor | None = None

        self.pos_weight: float | None = None

    def setup(self, stage: str | None = None):
        def _features(df: pd.DataFrame) -> pd.DataFrame:
            drop_cols = []
            if self.id_col in df.columns:
                drop_cols.append(self.id_col)
            if self.target_col in df.columns:
                drop_cols.append(self.target_col)
            return df.drop(columns=drop_cols, errors="ignore")

        if stage in ("fit", None):
            X_train_df = _features(self.df_train)
            X_val_df = _features(self.df_val)

            self.prep = TabPreprocessor(
                target_col=self.target_col,
                id_col=self.id_col,
                cat_cols=self.cat_cols,
                num_cols=self.num_cols,
                cat_unique_threshold=self.cat_unique_threshold,
            ).fit(self.df_train)

            X_num_tr, X_cat_tr = self.prep.transform(self.df_train)
            X_num_va, X_cat_va = self.prep.transform(self.df_val)

            y_tr = self.df_train[self.target_col].values.astype(np.float32)
            y_va = self.df_val[self.target_col].values.astype(np.float32)

            self.train_dataset = TabularDataset(X_num_tr, X_cat_tr, y_tr)
            self.val_dataset = TabularDataset(X_num_va, X_cat_va, y_va)

            pos = float(y_tr.sum())
            neg = float(len(y_tr) - pos)
            self.pos_weight = neg / max(pos, 1.0)

        if stage in ("test", None):
            if self.df_test is None:
                return

            X_num_te, X_cat_te = self.prep.transform(self.df_test) if self.prep is not None else (None, None)

            if self.target_col in self.df_test.columns:
                y_te = self.df_test[self.target_col].values.astype(np.float32)
                self.test_dataset = TabularDataset(X_num_te, X_cat_te, y_te)
            else:
                self.test_dataset = TabularDataset(X_num_te, X_cat_te, y=None)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
            drop_last=False,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            drop_last=False,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            drop_last=False,
        )


In [11]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
from torchmetrics import Accuracy, F1Score, AUROC

class SmokingTabularClassifier(pl.LightningModule):
    def __init__(
        self,
        cat_cardinalities: list[int],
        n_num: int,
        learning_rate: float = 1e-3,
        hidden: int = 256,
        dropout: float = 0.25,
        pos_weight: float | None = None,
        emb_max_dim: int = 16,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.n_cat = len(cat_cardinalities)
        self.n_num = n_num

        self.embs = nn.ModuleList()
        emb_total = 0
        for card in cat_cardinalities:
            emb_dim = min(emb_max_dim, max(2, int(round(card ** 0.25 * 4))))
            self.embs.append(nn.Embedding(card, emb_dim))
            emb_total += emb_dim

        in_dim = emb_total + n_num

        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden // 2, 1),
        )

        if pos_weight is not None:
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
        else:
            self.criterion = nn.BCEWithLogitsLoss()

        self.train_acc = Accuracy(task="binary")
        self.val_acc = Accuracy(task="binary")
        self.train_f1 = F1Score(task="binary")
        self.val_f1 = F1Score(task="binary")
        self.train_auc = AUROC(task="binary")
        self.val_auc = AUROC(task="binary")

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor):
        feats = []

        if self.n_cat > 0:
            embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
            feats.append(torch.cat(embs, dim=1))

        if self.n_num > 0:
            feats.append(x_num)

        x = torch.cat(feats, dim=1) if len(feats) > 1 else feats[0]
        logits = self.net(x).squeeze(1)
        return logits

    def _shared_step(self, batch, prefix: str):
        x_num, x_cat, y = batch
        y = y.float()

        logits = self(x_num, x_cat)
        loss = self.criterion(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).long()
        y_long = y.long()

        if prefix == "train":
            self.train_acc(preds, y_long)
            self.train_f1(preds, y_long)
            self.train_auc(probs, y_long)

            self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
            self.log("train_acc", self.train_acc, prog_bar=True, on_epoch=True, on_step=False)
            self.log("train_f1", self.train_f1, prog_bar=True, on_epoch=True, on_step=False)
            self.log("train_auc", self.train_auc, prog_bar=True, on_epoch=True, on_step=False)
        else:
            self.val_acc(preds, y_long)
            self.val_f1(preds, y_long)
            self.val_auc(probs, y_long)

            self.log("val_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
            self.log("val_acc", self.val_acc, prog_bar=True, on_epoch=True, on_step=False)
            self.log("val_f1", self.val_f1, prog_bar=True, on_epoch=True, on_step=False)
            self.log("val_auc", self.val_auc, prog_bar=True, on_epoch=True, on_step=False)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "val")

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=1e-2)


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

df = pd.read_csv('/content/train.csv')

df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["smoking"]
)

data_module = SmokingDataModule(
    df_train=df_train,
    df_val=df_val,
    df_test=None,
    batch_size=512,
)

data_module.setup("fit")

model = SmokingTabularClassifier(
    cat_cardinalities=data_module.prep.cat_cardinalities_,
    n_num=len(data_module.prep.num_cols),
    learning_rate=1e-3,
    hidden=256,
    dropout=0.25,
    pos_weight=data_module.pos_weight,
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_auc",
    mode="max",
    save_top_k=1,
    filename="best-checkpoint"
)

early_stop_callback = EarlyStopping(
    monitor="val_auc",
    patience=5,
    mode="max"
)

trainer = pl.Trainer(
    max_epochs=30,
    callbacks=[early_stop_callback, checkpoint_callback],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    log_every_n_steps=20
)

trainer.fit(model, data_module)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embs      │ ModuleList        │    768 │ train │     0 │
│ 1 │ net       │ Sequential        │  119 K │ train │     0 │
│ 2 │ criterion │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 3 │ train_acc │ BinaryAccuracy    │      0 │ train │     0 │
│ 4 │ val_acc   │ BinaryAccuracy    │      0 │ train │     0 │
│ 5 │ train_f1  │ BinaryF1Score     │      0 │ train │     0 │
│ 6 │ val_f1    │ BinaryF1Score     │      0 │ train │     0 │
│ 7 │ train_auc │ BinaryAUROC       │      0 │ train │     0 │
│ 8 │ val_auc   │ BinaryAUROC       │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 120 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 120 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [25]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

class InferenceTabDataset(Dataset):
    def __init__(self, X_num: np.ndarray | None, X_cat: np.ndarray | None):
        self.X_num = torch.from_numpy(X_num).float() if X_num is not None else None
        self.X_cat = torch.from_numpy(X_cat).long() if X_cat is not None else None

        if self.X_num is not None:
            self.n = self.X_num.shape[0]
        else:
            self.n = self.X_cat.shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x_num = self.X_num[idx] if self.X_num is not None else torch.empty(0, dtype=torch.float32)
        x_cat = self.X_cat[idx] if self.X_cat is not None else torch.empty(0, dtype=torch.long)
        return x_num, x_cat


@torch.no_grad()
def predict_test_submission(
    df_test: pd.DataFrame,
    checkpoint_path: str,
    preprocessor,
    batch_size: int = 2048,
    id_col: str = "id",
    out_path: str = "submission.csv",
):
    if preprocessor.num_cols:
        df_test = df_test.copy()
        for c in preprocessor.num_cols:
            if c in df_test.columns:
                df_test[c] = pd.to_numeric(df_test[c], errors="coerce")

    X_num, X_cat = preprocessor.transform(df_test)

    ds = InferenceTabDataset(X_num, X_cat)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SmokingFTTransformer.load_from_checkpoint(checkpoint_path)
    model.eval().to(device)

    probs_all = []
    for x_num, x_cat in dl:
        x_num = x_num.to(device)
        x_cat = x_cat.to(device)

        logits = model(x_num, x_cat)
        probs = torch.sigmoid(logits)
        probs_all.append(probs.detach().cpu().numpy())

    probs_all = np.concatenate(probs_all, axis=0)

    sub = pd.DataFrame({
        "id": df_test[id_col].values,
        "smoking": probs_all.astype(float),
    })
    sub.to_csv(out_path, index=False)
    return sub


In [14]:
df_test = pd.read_csv("/content/test.csv")

best_ckpt = checkpoint_callback.best_model_path
sub = predict_test_submission(
    df_test=df_test,
    checkpoint_path=best_ckpt,
    preprocessor=data_module.prep,
    out_path="submission.csv"
)

sub.head()


,id,smoking
0,15000,0.420541
1,15001,0.001715
2,15002,0.007737
3,15003,0.731207
4,15004,0.009795


In [19]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics import AUROC

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num: int, cat_cardinalities: list[int], d_model: int):
        super().__init__()
        self.n_num = n_num
        self.n_cat = len(cat_cardinalities)
        self.d_model = d_model

        if n_num > 0:
            self.num_w = nn.Parameter(torch.randn(n_num, d_model) * 0.02)
            self.num_b = nn.Parameter(torch.zeros(n_num, d_model))
        else:
            self.num_w = None
            self.num_b = None

        self.cat_embs = nn.ModuleList([nn.Embedding(card, d_model) for card in cat_cardinalities])

        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor):
        B = x_num.shape[0] if self.n_num > 0 else x_cat.shape[0]
        tokens = []

        if self.n_num > 0:
            xn = x_num.unsqueeze(-1)
            t_num = xn * self.num_w.unsqueeze(0) + self.num_b.unsqueeze(0)
            tokens.append(t_num)

        if self.n_cat > 0:
            t_cats = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embs)]
            t_cat = torch.stack(t_cats, dim=1)
            tokens.append(t_cat)

        x = torch.cat(tokens, dim=1) if len(tokens) > 1 else tokens[0]
        x = self.ln(x)

        cls = self.cls.expand(B, -1, -1)
        return torch.cat([cls, x], dim=1)


class SmokingFTTransformer(pl.LightningModule):
    def __init__(
        self,
        cat_cardinalities: list[int],
        n_num: int,
        d_model: int = 192,
        n_heads: int = 8,
        n_layers: int = 3,
        dropout: float = 0.1,
        learning_rate: float = 3e-4,
        pos_weight: float | None = None,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.tokenizer = FeatureTokenizer(n_num, cat_cardinalities, d_model)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )

        if pos_weight is not None:
            self.register_buffer("pos_w", torch.tensor(float(pos_weight)))
        else:
            self.pos_w = None

        self.val_auc = AUROC(task="binary")

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor):
        tok = self.tokenizer(x_num, x_cat)
        out = self.encoder(tok)
        cls = out[:, 0, :]
        logits = self.head(cls).squeeze(1)
        return logits

    def training_step(self, batch, batch_idx):
        x_num, x_cat, y = batch
        y = y.float()
        logits = self(x_num, x_cat)
        loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=self.pos_w)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, batch_idx):
        x_num, x_cat, y = batch
        y = y.long()
        logits = self(x_num, x_cat)
        probs = torch.sigmoid(logits)
        self.val_auc(probs, y)
        self.log("val_auc", self.val_auc, prog_bar=True, on_epoch=True, on_step=False)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=1e-2)


In [20]:
import pandas as pd
import torch
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

df = pd.read_csv("/content/train.csv")

df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["smoking"],
)

data_module = SmokingDataModule(
    df_train=df_train,
    df_val=df_val,
    df_test=None,
    batch_size=512,
)

data_module.setup("fit")


In [23]:
model = SmokingFTTransformer(
    cat_cardinalities=data_module.prep.cat_cardinalities_,
    n_num=len(data_module.prep.num_cols),
    pos_weight=data_module.pos_weight,
    d_model=192,
    n_layers=3,
    n_heads=8,
    dropout=0.1,
    learning_rate=3e-4,
)


checkpoint_callback = ModelCheckpoint(
    monitor="val_auc",
    mode="max",
    save_top_k=1,
    filename="ft-best-auc"
)

early_stop_callback = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6
)


trainer = pl.Trainer(
    max_epochs=50,
    callbacks=[checkpoint_callback, early_stop_callback],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    log_every_n_steps=50,
    enable_progress_bar=True,
)

trainer.fit(model, data_module)

best_ckpt = checkpoint_callback.best_model_path
print("Best checkpoint:", best_ckpt)



INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ tokenizer │ FeatureTokenizer   │ 24.8 K │ train │     0 │
│ 1 │ encoder   │ TransformerEncoder │  1.3 M │ train │     0 │
│ 2 │ head      │ Sequential         │    577 │ train │     0 │
│ 3 │ val_auc   │ BinaryAUROC        │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 49                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches 
(24) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if
you want to see logs for the training epoch.

Best checkpoint: /content/lightning_logs/version_16/checkpoints/ft-best-auc.ckpt


In [27]:
df_test = pd.read_csv("/content/test.csv")

sub = predict_test_submission(
    df_test=df_test,
    checkpoint_path=best_ckpt,
    preprocessor=data_module.prep,
    out_path="submission.csv"
)

sub.head()


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


,id,smoking
0,15000,0.328305
1,15001,0.019618
2,15002,0.107964
3,15003,0.364838
4,15004,0.009145
